
# TLC Trip Record Data

## Yellow Taxi Trip Records 2026 (January - )


https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page


https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf

In [40]:
import pandas as pd
import pyarrow
import numpy as np

### Data Cleaning Notes

renamed several columns to improve clarity. 
- tpep_pickup_datetime: pickup_datetime 
- tpep_dropoff_datetime" : dropoff_datetime
- extra : extra_charges
- trip_distance : trip_distance_miles

replaced strange unusable 0 values in columns where that's impossible, with Nan.
- passenger count

I noticed impossibly high distances recorded so i grabbed the largeest and compared them to the fares and noticed no consitency proving that that this column is faulty and contains errors I believe came from data collection. My theory is that these values werent necesarrly entered wrong but that the taximeter within that cab at that time or was faulty and recording incorrect information. 

I plan to check and see if i can find any other overlap between these outliers including 


of this dataset of over 3 million taxi trips only 162 have trips have trips recorded to be over 100 miles and 620 over 50 miles. 

- PULocationID	DOLocationID and fare_amount are a better measure of the length of the ride (taxi zone look up csv on tlc site)


### Data Analysis
 ideas
- check for overlap on vendor id or other identifers like time of year for any specifc types of errors(like the passenger count or trp distance outliers)
- Which day generated the highest total revenue?
- How much money is generated by each payment type?
- What are the busiest pickup hours?
- Do larger groups travel farther?
- Does passenger count affect tip percentage?

visualizations 
- What percentage of trips use each payment method?
- Revenue by Hour


In [64]:

file = "/Users/kamari/Documents/project_info/yellow_tripdata_2026-JAN.parquet"

zone_lookup= pd.read_csv("/Users/kamari/Documents/project_info/taxi_zone_lookup.csv")

taxi_zones = pd.DataFrame(zone_lookup)


data = pd.read_parquet(file)
df = pd.DataFrame(data)

df.dtypes

VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag                  str
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

In [72]:
yellow_taxi = df.copy()

In [73]:
yellow_taxi.rename(columns={
    "tpep_pickup_datetime": "pickup_datetime" , 
    "tpep_dropoff_datetime" : "dropoff_datetime",
    "extra" : "extra_charges",
    "trip_distance" : "trip_distance_miles",
}, inplace=True)


In [74]:
yellow_taxi.drop(columns=['store_and_fwd_flag', 'improvement_surcharge'], inplace=True)

In [75]:
yellow_taxi['passenger_count'] = yellow_taxi['passenger_count'].replace(0, np.nan)
yellow_taxi


,VendorID,pickup_datetime,dropoff_datetime,passenger_count,trip_distance_miles,RatecodeID,PULocationID,DOLocationID,payment_type,fare_amount,extra_charges,mta_tax,tip_amount,tolls_amount,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,239,238,1,7.20,1.00,0.5,3.66,0.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,NaN,0.90,1.0,163,162,2,7.90,4.25,0.5,0.00,0.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,NaN,1.40,1.0,43,237,1,10.70,4.25,0.5,2.50,0.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,142,209,1,38.70,1.00,0.5,11.11,0.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,NaN,2.16,1.0,88,144,1,13.50,1.00,0.5,3.85,0.0,23.10,2.5,0.0,0.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3724884,2,2026-01-31 23:26:00,2026-01-31 23:39:16,NaN,1.62,NaN,237,161,0,17.09,0.00,0.5,0.00,0.0,21.84,NaN,NaN,0.75
3724885,2,2026-01-31 23:33:53,2026-01-31 23:34:07,NaN,0.00,NaN,42,42,0,23.19,0.00,0.5,0.00,0.0,24.69,NaN,NaN,0.00
3724886,2,2026-01-31 23:40:23,2026-01-31 23:56:10,NaN,6.84,NaN,137,69,0,29.21,0.00,0.5,0.00,0.0,33.96,NaN,NaN,0.75
3724887,2,2026-01-31 23:10:21,2026-01-31 23:20:00,NaN,1.53,NaN,137,162,0,19.19,0.00,0.5,0.00,0.0,23.94,NaN,NaN,0.75


In [56]:
yellow_taxi["trip_distance_miles"].describe()

count    3.724889e+06
mean     6.455647e+00
std      6.488855e+02
min      0.000000e+00
25%      1.000000e+00
50%      1.810000e+00
75%      3.730000e+00
max      2.690975e+05
Name: trip_distance_miles, dtype: float64

In [63]:
yellow_taxi.isnull().sum()

VendorID                       0
pickup_datetime                0
dropoff_datetime               0
passenger_count          1102845
trip_distance_miles            0
RatecodeID               1088058
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra_charges                  0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1088058
Airport_fee              1088058
cbd_congestion_fee             0
dtype: int64

In [68]:
yellow_taxi.loc[
    yellow_taxi["trip_distance_miles"].nlargest(20).index,
    ["VendorID", "trip_distance_miles", "fare_amount"]
]

,VendorID,trip_distance_miles,fare_amount
3524923,2,269097.48,12.19
3632367,2,237567.19,72.99
3285750,2,236216.69,13.58
3131676,2,236183.06,22.75
2704619,2,221544.91,31.21
2688274,2,221436.73,27.61
3131406,2,196490.48,33.59
3571687,2,192081.68,60.20
2820951,2,184908.33,12.18
3499856,2,183363.75,71.01


In [ ]:
(yellow_taxi["trip_distance_miles"] > 50).sum()

(yellow_taxi["trip_distance_miles"] > 100).sum()

np.int64(620)

In [65]:
taxi_zones

,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
...,...,...,...,...
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,NaN,NaN
